In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Synthetic dataset
n = 1000
X = pd.DataFrame({
    'monthly_spend': np.random.exponential(120, n),
    'last_login_days': np.random.exponential(30, n),
})
y = np.random.randint(0, 2, n)

print(f"Dataset shape: {X.shape}")
print(f"\nmonthly_spend stats:")
print(X['monthly_spend'].describe())

## Bug 1: fit_transform on full dataset before split (data leakage)

In [ ]:
# --- BUGGY CODE ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fitted on FULL dataset

# Split AFTER scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Scaler mean (from full dataset): {scaler.mean_[0]:.2f}")
print(f"Training set mean (before scaling): {X.iloc[:int(len(X)*0.8)]['monthly_spend'].mean():.2f}")
print(f"Test set mean (before scaling): {X.iloc[int(len(X)*0.8):]['monthly_spend'].mean():.2f}")

**Explanation:** The scaler learned mean/std from BOTH training and test data. Write here why this is problematic for model evaluation.

**Fix:**

In [ ]:
# Correct: split FIRST, fit ONLY on training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # transform only

print(f"Scaler mean (from training only): {scaler.mean_[0]:.2f}")
print(f"Training set mean (before scaling): {X_train['monthly_spend'].mean():.2f}")
print(f"Match: True (scaler was fit on training data)")

## Bug 2: fit_transform on test set instead of transform

In [ ]:
# --- BUGGY CODE ---
scaler2 = StandardScaler()
X_train_scaled2 = scaler2.fit_transform(X_train)
X_test_scaled2 = scaler2.fit_transform(X_test)  # BUG: second fit_transform

print(f"Training mean: {X_train['monthly_spend'].mean():.2f}")
print(f"Scaler was fit on test data, so scaler.mean_: {scaler2.mean_[0]:.2f}")
print(f"\nExplanation: Write here why this breaks evaluation.")

**Fix:**

In [ ]:
# Use transform only on test
scaler3 = StandardScaler()
X_train_scaled3 = scaler3.fit_transform(X_train)
X_test_scaled3 = scaler3.transform(X_test)  # FIXED: transform, not fit_transform

print(f"Training and test now use the same scaler (fit on training data)")
print(f"Evaluation is now valid.")

## Bug 3: pd.cut (equal-width) instead of pd.qcut (equal-frequency) binning

In [ ]:
# --- BUGGY CODE ---
df_test = pd.DataFrame({
    'last_login_days': X_test['last_login_days'].values
}).reset_index(drop=True)

df_test['recency_band'] = pd.cut(
    df_test['last_login_days'],
    bins=4,
    labels=['active', 'recent', 'dormant', 'inactive']
)

print(df_test['recency_band'].value_counts().sort_index())
print(f"\nExplanation: Write here what's wrong with these unbalanced bins.")

**Fix:**

In [ ]:
# Use qcut for equal-frequency bins
df_test['recency_band'] = pd.qcut(
    df_test['last_login_days'],
    q=4,
    labels=['active', 'recent', 'dormant', 'inactive'],
    duplicates='drop'  # Handle cases where quantiles are tied
)

print("After fix (roughly equal frequency):")
print(df_test['recency_band'].value_counts().sort_index())
print("Each bin now has similar number of customers.")

## Summary check

All three bugs fixed:

In [ ]:
print("Bug 1: fit_transform on full data → split first, fit on train only")
print("Bug 2: fit_transform on test → use transform on test")
print("Bug 3: pd.cut (equal-width) → pd.qcut (equal-frequency)")
print("\nAll fixed. Ready for the next lesson on datetime and text features.")